# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides an example of loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print dataset metadata name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Let's enumerate all record sets and print their `@id`, available fields, and columns.

In [ ]:
# Show all record sets (by @id)
record_sets = list(dataset.metadata.record_set or [])
if len(record_sets) == 0:
    print("No record sets defined in the dataset schema. Attempting to infer from the dataset...")
    # Try to discover possible record_set_ids via dataset.record_sets()
    recordset_ids = dataset.record_sets()
    print(f"Available record sets in this dataset: {recordset_ids}")
    all_ids = recordset_ids
else:
    all_ids = [rs['@id'] for rs in record_sets]
    print(f"Available record sets in metadata: {all_ids}")

if not all_ids:
    raise ValueError('No record sets found.')

# For each record set, print out the available fields (by @id)
for record_set_id in all_ids:
    print(f"\nRecord Set: {record_set_id}")
    try:
        recordset_metadata = dataset.record_set(record_set_id)
        # print fields (@id)
        if hasattr(recordset_metadata, 'field') and recordset_metadata.field:
            field_ids = [field['@id'] for field in recordset_metadata.field]
            print(f"  Fields: {field_ids}")
        else:
            print("  Fields: Not available in schema.")
        # print columns (@id)
        if hasattr(recordset_metadata, 'column') and recordset_metadata.column:
            column_ids = [col['@id'] for col in recordset_metadata.column]
            print(f"  Columns: {column_ids}")
        else:
            print("  Columns: Not available in schema.")
    except Exception as e:
        print(f"  (Could not retrieve metadata for this record set: {e})")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we attempt to load records for each record set.

In [ ]:
# List of discovered or selected record set @ids
record_sets_to_load = all_ids
dataframes = {}

for record_set_id in record_sets_to_load:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for Record Set {record_set_id}. Columns:")
            print(df.columns.tolist())
        else:
            print(f"No records found for Record Set {record_set_id}.")
    except Exception as e:
        print(f"Error loading records for Record Set {record_set_id}: {e}")

# For demonstration, pick the first available dataframe
if dataframes:
    preview_id = list(dataframes.keys())[0]
    print(f"\nPreviewing data from Record Set {preview_id}:\n")
    display(dataframes[preview_id].head())
else:
    raise ValueError("No dataframes could be loaded from the available record sets.")

## 4. Exploratory Data Analysis (EDA)

Now let's try basic filtering, normalization, and grouping on the loaded data. We'll select a numeric field (for demonstration, you may need to adjust to a valid numeric field available in the chosen record set), filter by a threshold, normalize, and optionally show a groupby aggregation.

In [ ]:
# Select record set for further analysis (first loaded)
record_set_id = preview_id
df = dataframes[record_set_id]
print(f"Working with Record Set: {record_set_id}")

# Choose a numeric field; list all columns for user review
print("Available columns:", df.columns.tolist())
# For demonstration, choose one column likely to be numeric (adjust as needed)
possible_numeric_cols = [col for col in df.columns if df[col].dtype in [int, float, 'int64', 'float64', 'float32']]
if not possible_numeric_cols:
    # Fallback: try to coerce columns to numeric
    candidates = []
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].notna().sum() > 0:
                candidates.append(col)
        except Exception:
            continue
    possible_numeric_cols = candidates

if not possible_numeric_cols:
    raise ValueError("No numeric fields found for EDA.")

# Use the first found numeric field for demonstration
numeric_field = possible_numeric_cols[0]
threshold = df[numeric_field].mean()

filtered_df = df[df[numeric_field] > threshold]
print(f"\nFiltered records with {numeric_field} > {threshold:.3f}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to find a grouping/categorical field
possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
group_field = possible_group_fields[0] if possible_group_fields else None

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean').reset_index()
    print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the selected numeric field, and if a grouping field is available, show a comparison by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True, color='skyblue')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if group_field is not None:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We loaded the FAIR² dataset Croissant schema and explored its metadata and available record sets by `@id`.
- Data was loaded into pandas DataFrames for each record set. Numeric fields were filtered, normalized, and groupwise statistics computed.
- Initial visualizations give insight into data distributions and variability across groups, supporting subsequent data analysis workflows.

_For further analysis, refer to the full Croissant schema to map field `@id`s contextually to your research or application domain._